## Пешеходные изохроны 15 мин

In [35]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
# Эта строка отключает отображение всех предупреждений DeprecationWarning
warnings.filterwarnings('ignore', category=DeprecationWarning)
# --- 1. ПОДГОТОВКА ЕДИНОГО ДАТАФРЕЙМА ---

# Список файлов и их типов
files = [
    ('poi/beauty.csv', 'beauty'),
    ('poi/bus.csv', 'bus'),
    ('poi/cafe.csv', 'cafe'),
    ('poi/cafe_checks.csv', 'cafe_checks'),
    ('poi/coffee.csv', 'coffee'),
    ('poi/dental.csv', 'dental'),
    ('poi/hospital.csv', 'hospital'),
    ('poi/metro.csv', 'metro'),
    ('poi/molls.csv', 'molls'),
    ('poi/parks.csv', 'parks'),
    ('poi/pharma.csv', 'pharma'),
    ('poi/study.csv', 'study'),
    ('poi/supermarkets.csv', 'supermarkets'),
    ('poi/supermarkets4rich.csv', 'supermarkets4rich'),
    ('poi/workplaces.csv', 'workplaces')
]

# Список для сбора кусочков
all_pieces = []

for file, poi_type in files:
    print(f"Загружаем {file}...")
    
    # Читаем CSV
    df = pd.read_csv(file)
    
    # Проверяем наличие координат
    if not {'lat', 'lon'}.issubset(df.columns):
        print(f"Внимание: в {file} нет колонок lat/lon. Пропускаем.")
        continue
    
    # Создаем GeoDataFrame и добавляем тип точки
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lon, df.lat),
        crs="EPSG:4326"
    )
    
    # Добавляем колонку с типом POI для группировки
    gdf['poi_type'] = poi_type
    
    all_pieces.append(gdf)

# Объединяем все кусочки в один большой GeoDataFrame
gdf_all = gpd.GeoDataFrame(pd.concat(all_pieces, ignore_index=True), crs="EPSG:4326")
print("Единый датафрейм создан.")
print(f"Всего точек: {len(gdf_all)}")
fitness = pd.read_csv('Фитнес/fitness_w_centerd.csv')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами

WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 15

# --- ЗАГРУЗКА ГРАФА (ОДИН РАЗ) ---
print("Загружаем пешеходный граф Москвы...")
# G_walk = ox.load_graphml(f'{GRAPH_PATH}/moscow_walk.graphml')
# G_walk = ox.project_graph(G_walk) # Проекция в метры (UTM)
# current_crs = G_walk.graph['crs']

start = time.time()  # или time.perf_counter()

# Проецируем офисы в ту же систему координат, что и граф
gdf_all = gdf_all.to_crs(current_crs)

# Добавляем атрибут времени на ребра графа (если его еще нет)
# Это нужно сделать один раз для всего графа, а не в цикле
for u, v, k, data in G_walk.edges(keys=True, data=True):
    # Проверяем наличие атрибута длины и приводим к float на всякий случай
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN

results = []
print(f"Начинаем обработку {len(fitness)} объектов...")


for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 2 == 0:
        print(f"Обрабатываем фитнес-клуб {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_walk, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_walk, center_node, radius=WALK_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась, записываем нули для всех типов
            empty_row = {'fitness_id': idx}
            for t in gdf_all['poi_type'].unique():
                empty_row[f'{t}_count'] = 0
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ (ГЛАВНАЯ ОПТИМИЗАЦИЯ) ---
        # Ищем все точки из ЕДИНОГО датафрейма, попавшие в полигон
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_all, how='inner', predicate='intersects')
        
        # Если ничего не найдено, записываем нули
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for t in gdf_all['poi_type'].unique():
                empty_row[f'{t}_count'] = 0
            results.append(empty_row)
            continue

# --- ГРУППИРОВКА И РАСЧЕТ МЕТРИК ---
        # Создаем словарь для хранения результатов по текущему фитнесу
        row_result = {'fitness_id': idx}
        
        # Группируем найденные точки по их типу
        grouped = hits.groupby('poi_type')

        for name, group in grouped:
            # 1. Считаем количество точек для каждого типа
            row_result[f'{name}_count'] = len(group)

            # 2. Считаем метрики для типов с рейтингами
            if name in ['beauty', 'cafe', 'coffee', 'dental', 'hospital', 
                        'parks', 'pharma', 'study', 'supermarkets', 
                        'supermarkets4rich', 'workplaces', 'molls']:
                # Проверяем, есть ли в этой группе колонка с отзывами и есть ли ненулевые значения
                if 'rating_cnt' in group.columns and group['rating_cnt'].notnull().any():
                    # .sum() и .mean() в pandas автоматически игнорируют NaN
                    row_result[f'{name}_reviews_sum'] = group['rating_cnt'].sum()
                    row_result[f'{name}_reviews_mean'] = group['rating_cnt'].mean()

            # 3. Специфичная логика для cafe_checks (уникальные точки)
            if name == 'cafe_checks':
                # Группируем по координатам, чтобы получить уникальные кафе
                # Используем round(5) для сглаживания погрешностей координат
                unique_cafes = group.copy()
                unique_cafes['lat_round'] = unique_cafes['lat'].round(5)
                unique_cafes['lon_round'] = unique_cafes['lon'].round(5)
                unique_cafes = unique_cafes.groupby(['lat_round', 'lon_round'], as_index=False).first()

                # Считаем количество УНИКАЛЬНЫХ кафе
                row_result['cafe_checks_count'] = len(unique_cafes)

                # Считаем метрики только по уникальным записям
                if 'avg_price' in unique_cafes.columns and unique_cafes['avg_price'].notnull().any():
                    valid_prices = unique_cafes['avg_price'].dropna()
                    if len(valid_prices) > 0:
                        row_result['cafe_checks_avg_price_mean'] = valid_prices.mean()
                        row_result['cafe_checks_avg_price_median'] = valid_prices.median()

                if 'capacity' in unique_cafes.columns and unique_cafes['capacity'].notnull().any():
                    valid_capacity = unique_cafes['capacity'].dropna()
                    if len(valid_capacity) > 0:
                        row_result['cafe_checks_capacity_mean'] = valid_capacity.mean()
                        row_result['cafe_checks_capacity_median'] = valid_capacity.median()

            # 4. Специфичная логика для molls
            if name == 'molls':
                if 'branches_in_moll' in group.columns and group['branches_in_moll'].notnull().any():
                    row_result['molls_branches_sum'] = group['branches_in_moll'].sum()
                    row_result['molls_branches_mean'] = group['branches_in_moll'].mean()
                
                if 'building_floors' in group.columns and group['building_floors'].notnull().any():
                    row_result['molls_floors_sum'] = group['building_floors'].sum()
                    row_result['molls_floors_mean'] = group['building_floors'].mean()
                
                if 'parking' in group.columns and group['parking'].notnull().any():
                    row_result['molls_parking_sum'] = group['parking'].sum()
                    row_result['molls_parking_mean'] = group['parking'].mean()
        
        # Добавляем собранный результат в общий список
        results.append(row_result)
        

    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки тоже записываем нули для всех метрик, чтобы не ломать структуру финального датафрейма
        empty_row = {'fitness_id': idx}
        for t in gdf_all['poi_type'].unique():
            empty_row[f'{t}_count'] = 0.0
            empty_row[f'{t}_reviews_sum'] = 0.0
            empty_row[f'{t}_reviews_mean'] = 0.0
        results.append(empty_row)

Загружаем poi/beauty.csv...
Загружаем poi/bus.csv...
Загружаем poi/cafe.csv...
Загружаем poi/cafe_checks.csv...
Загружаем poi/coffee.csv...
Загружаем poi/dental.csv...
Загружаем poi/hospital.csv...
Загружаем poi/metro.csv...
Загружаем poi/molls.csv...
Загружаем poi/parks.csv...
Загружаем poi/pharma.csv...
Загружаем poi/study.csv...
Загружаем poi/supermarkets.csv...
Загружаем poi/supermarkets4rich.csv...
Загружаем poi/workplaces.csv...
Единый датафрейм создан.
Всего точек: 72436
Загружаем пешеходный граф Москвы...
Начинаем обработку 5029 объектов...
Обрабатываем фитнес-клуб 1/5029...
Обрабатываем фитнес-клуб 251/5029...
Обрабатываем фитнес-клуб 501/5029...
Обрабатываем фитнес-клуб 751/5029...
Обрабатываем фитнес-клуб 1001/5029...
Обрабатываем фитнес-клуб 1251/5029...
Обрабатываем фитнес-клуб 1501/5029...
Обрабатываем фитнес-клуб 1751/5029...
Обрабатываем фитнес-клуб 2001/5029...
Обрабатываем фитнес-клуб 2251/5029...
Обрабатываем фитнес-клуб 2501/5029...
Обрабатываем фитнес-клуб 2751/502

In [36]:
res_df = pd.DataFrame(results)
print("Готово!")
end = time.time()
print(f"Время: {end - start:.4f} сек")
fitness_iso_poi = pd.concat([fitness,res_df], axis=1)
fitness_iso_poi.to_csv('fitness_poi_iso.csv', index = False)
fitness_iso_poi.to_excel('fitness_poi_iso.xlsx', index = False)

Готово!
Время: 25510.6453 сек


In [42]:
19000/60/60

5.277777777777778

In [24]:
183/50*5100/60/60

5.1850000000000005

In [38]:
fitness_iso_poi

,club_key,first_seen,last_seen,observations_count,is_closed,is_new,lifespan_days,clean_title,clean_address,clean_district,...,supermarkets_reviews_mean,supermarkets4rich_count,supermarkets4rich_reviews_sum,supermarkets4rich_reviews_mean,workplaces_count,workplaces_reviews_sum,workplaces_reviews_mean,parks_count,parks_reviews_sum,parks_reviews_mean
0,#methodksu_улицамакаренко5ст1а,2024-07-25,2025-02-07,3,True,True,197,#methodksu,улицамакаренко5ст1а,басманный,...,43.100000,8.0,300.0,37.500000,20.0,320.0,18.823529,NaN,NaN,NaN
1,#sekta_3яулицаямскогополя22ст3,2024-02-22,2024-02-22,1,True,False,0,#sekta,3яулицаямскогополя22ст3,беговой,...,20.941176,8.0,62.0,7.750000,13.0,171.0,15.545455,NaN,NaN,NaN
2,#slimfitclub_улицалужники24ст48,2024-02-22,2026-03-24,8,False,False,761,#slimfitclub,улицалужники24ст48,хамовники,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,#бойкийфитнес_улицаалександрымонаховой88к3,2024-02-22,2025-11-17,6,True,False,634,#бойкийфитнес,улицаалександрымонаховой88к3,коммунарка,...,32.642857,5.0,86.0,21.500000,2.0,7.0,7.000000,NaN,NaN,NaN
4,#какорех_1йдербеневскийпереулок5,2024-02-22,2024-12-13,3,True,False,295,#какорех,1йдербеневскийпереулок5,даниловский,...,31.666667,1.0,2.0,2.000000,21.0,284.0,17.750000,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5024,ярядом_боровскоешоссе2к5,2024-07-25,2024-07-25,1,True,True,0,ярядом,боровскоешоссе2к5,солнцево,...,27.363636,3.0,63.0,21.000000,5.0,20.0,6.666667,NaN,NaN,NaN
5025,ярядом_улицапроизводственная8к2,2024-07-25,2024-07-25,1,True,True,0,ярядом,улицапроизводственная8к2,солнцево,...,29.842105,3.0,57.0,19.000000,2.0,30.0,15.000000,NaN,NaN,NaN
5026,ясенево_ясногорскаяулица5,2024-07-25,2024-07-25,1,True,True,0,ясенево,ясногорскаяулица5,ясенево,...,42.777778,3.0,19.0,6.333333,10.0,431.0,53.875000,NaN,NaN,NaN
5027,ясный_новоясеневскийпроспект24к4,2024-07-25,2024-07-25,1,True,True,0,ясный,новоясеневскийпроспект24к4,ясенево,...,39.800000,6.0,116.0,19.333333,12.0,426.0,60.857143,NaN,NaN,NaN


In [49]:
columns_to_rename = [
       'beauty_count', 'beauty_reviews_sum', 'beauty_reviews_mean',
       'bus_count', 'cafe_count', 'cafe_reviews_sum', 'cafe_reviews_mean',
       'cafe_checks_count', 'cafe_checks_avg_price_mean',
       'cafe_checks_avg_price_median', 'cafe_checks_capacity_mean',
       'cafe_checks_capacity_median', 'coffee_count', 'coffee_reviews_sum',
       'coffee_reviews_mean', 'dental_count', 'dental_reviews_sum',
       'dental_reviews_mean', 'hospital_count', 'hospital_reviews_sum',
       'hospital_reviews_mean', 'metro_count', 'molls_count',
       'molls_reviews_sum', 'molls_reviews_mean', 'molls_branches_sum',
       'molls_branches_mean', 'molls_floors_sum', 'molls_floors_mean',
       'molls_parking_sum', 'molls_parking_mean', 'pharma_count',
       'pharma_reviews_sum', 'pharma_reviews_mean', 'study_count',
       'study_reviews_sum', 'study_reviews_mean', 'supermarkets_count',
       'supermarkets_reviews_sum', 'supermarkets_reviews_mean',
       'supermarkets4rich_count', 'supermarkets4rich_reviews_sum',
       'supermarkets4rich_reviews_mean', 'workplaces_count',
       'workplaces_reviews_sum', 'workplaces_reviews_mean', 'parks_count',
       'parks_reviews_sum', 'parks_reviews_mean'
]

# Создаем словарь для переименования: {старое_имя: новое_имя}
rename_dict = {col: f"{col}_15ped" for col in columns_to_rename}

# Применяем переименование к датафрейму (изменения вносятся в исходный датафрейм)
# fitness_iso_poi.rename(columns=rename_dict, inplace=True)

# Для создания нового датафрейма без изменения оригинала используйте:
new_df = fitness_iso_poi.rename(columns=rename_dict)

In [50]:
new_df.columns[60:]

Index(['parking', 'free_parking', 'payed_parking', 'nearest_station',
       'nearest_station_type', 'nearest_station_d', 'count_stations',
       'all_stations', 'object_floor', 'floor_min', 'floor_max', 'floor_count',
       'dist_to_center_drive_m', 'dist_to_center_walk_m', 'fitness_id',
       'beauty_count_15ped', 'beauty_reviews_sum_15ped',
       'beauty_reviews_mean_15ped', 'bus_count_15ped', 'cafe_count_15ped',
       'cafe_reviews_sum_15ped', 'cafe_reviews_mean_15ped',
       'cafe_checks_count_15ped', 'cafe_checks_avg_price_mean_15ped',
       'cafe_checks_avg_price_median_15ped', 'cafe_checks_capacity_mean_15ped',
       'cafe_checks_capacity_median_15ped', 'coffee_count_15ped',
       'coffee_reviews_sum_15ped', 'coffee_reviews_mean_15ped',
       'dental_count_15ped', 'dental_reviews_sum_15ped',
       'dental_reviews_mean_15ped', 'hospital_count_15ped',
       'hospital_reviews_sum_15ped', 'hospital_reviews_mean_15ped',
       'metro_count_15ped', 'molls_count_15ped', 'm

In [40]:
res_df.describe()

,fitness_id,beauty_count,beauty_reviews_sum,beauty_reviews_mean,bus_count,cafe_count,cafe_reviews_sum,cafe_reviews_mean,cafe_checks_count,cafe_checks_avg_price_mean,...,supermarkets_reviews_mean,supermarkets4rich_count,supermarkets4rich_reviews_sum,supermarkets4rich_reviews_mean,workplaces_count,workplaces_reviews_sum,workplaces_reviews_mean,parks_count,parks_reviews_sum,parks_reviews_mean
count,5029.000000,4752.000000,4711.000000,4711.000000,4770.000000,4749.000000,4709.000000,4709.000000,4380.000000,4318.000000,...,4696.000000,4441.000000,4390.000000,4390.000000,4197.000000,4092.000000,4092.000000,672.000000,585.000000,585.000000
mean,2514.000000,73.638889,1926.734451,32.570280,17.806080,38.576964,3473.139308,60.300280,21.168493,1050.043139,...,39.836482,4.251745,84.015034,20.166341,10.838456,261.370235,22.562108,1.093750,282.441026,236.811396
std,1451.891582,52.619119,1775.405828,15.600287,7.797118,59.535016,7575.854323,46.427156,24.742186,1560.213939,...,24.307681,2.595932,81.179190,24.189439,10.286789,406.356468,22.631071,0.442008,569.325878,471.808848
min,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,200.000000,...,2.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000
25%,1257.000000,38.000000,668.000000,22.421053,12.000000,10.000000,257.000000,25.200000,7.000000,783.461538,...,27.444444,2.000000,30.000000,10.500000,3.000000,27.000000,9.000000,1.000000,37.000000,35.000000
50%,2514.000000,64.000000,1482.000000,29.986301,18.000000,19.000000,760.000000,46.666667,13.000000,912.500000,...,33.885621,4.000000,62.000000,17.000000,7.000000,124.000000,18.205263,1.000000,79.000000,79.000000
75%,3771.000000,97.000000,2509.500000,40.205263,23.000000,37.000000,2627.000000,85.213675,25.000000,1050.000000,...,43.700658,6.000000,119.000000,24.800000,16.000000,346.000000,28.450000,1.000000,219.000000,166.500000
max,5028.000000,445.000000,11740.000000,171.684211,49.000000,448.000000,59034.000000,361.333333,181.000000,39727.777778,...,279.000000,15.000000,864.000000,798.000000,66.000000,3477.000000,366.000000,3.000000,2732.000000,2731.000000


In [33]:
res_df.columns

Index(['fitness_id', 'beauty_count', 'beauty_reviews_sum',
       'beauty_reviews_mean', 'bus_count', 'cafe_count', 'cafe_reviews_sum',
       'cafe_reviews_mean', 'coffee_count', 'coffee_reviews_sum',
       'coffee_reviews_mean', 'dental_count', 'dental_reviews_sum',
       'dental_reviews_mean', 'hospital_count', 'hospital_reviews_sum',
       'hospital_reviews_mean', 'pharma_count', 'pharma_reviews_sum',
       'pharma_reviews_mean', 'study_count', 'study_reviews_sum',
       'study_reviews_mean', 'supermarkets_count', 'supermarkets_reviews_sum',
       'supermarkets_reviews_mean', 'supermarkets4rich_count',
       'supermarkets4rich_reviews_sum', 'supermarkets4rich_reviews_mean',
       'workplaces_count', 'workplaces_reviews_sum', 'workplaces_reviews_mean',
       'cafe_checks_count', 'cafe_checks_avg_price_mean',
       'cafe_checks_avg_price_median', 'cafe_checks_capacity_mean',
       'cafe_checks_capacity_median', 'metro_count', 'molls_count',
       'molls_reviews_sum', 'mol

In [41]:
res_df[['bus_count','metro_count', 'molls_count',
       'molls_reviews_sum', 'molls_reviews_mean', 'molls_branches_sum']].describe()

,bus_count,metro_count,molls_count,molls_reviews_sum,molls_reviews_mean,molls_branches_sum
count,4770.000000,3342.000000,1694.000000,1662.000000,1662.000000,1653.000000
mean,17.806080,1.945841,1.172373,323.737665,267.374549,213.212341
std,7.797118,1.430124,0.480983,532.696669,415.717948,180.114650
min,0.000000,0.000000,0.000000,4.000000,4.000000,1.000000
25%,12.000000,1.000000,1.000000,48.000000,43.000000,81.000000
50%,18.000000,1.000000,1.000000,128.000000,110.000000,171.000000
75%,23.000000,2.000000,1.000000,285.000000,273.000000,284.000000
max,49.000000,12.000000,3.000000,4607.000000,2303.500000,928.000000


In [51]:
new_df.to_csv('fitness_poi_iso15ped.csv', index = False)
new_df.to_excel('fitness_poi_iso15ped.xlsx', index = False)

## Пешеходные изохроны 5 мин

In [59]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
# Эта строка отключает отображение всех предупреждений DeprecationWarning
warnings.filterwarnings('ignore', category=DeprecationWarning)
# --- 1. ПОДГОТОВКА ЕДИНОГО ДАТАФРЕЙМА ---

# Список файлов и их типов
files = [
    ('poi/beauty.csv', 'beauty'),
    ('poi/bus.csv', 'bus'),
    ('poi/cafe.csv', 'cafe'),
    # ('poi/cafe_checks.csv', 'cafe_checks'),
    ('poi/coffee.csv', 'coffee'),
    # ('poi/dental.csv', 'dental'),
    # ('poi/hospital.csv', 'hospital'),
    ('poi/metro.csv', 'metro'),
    ('poi/molls.csv', 'molls'),
    # ('poi/parks.csv', 'parks'),
    # ('poi/pharma.csv', 'pharma'),
    # ('poi/study.csv', 'study'),
    # ('poi/supermarkets.csv', 'supermarkets'),
    ('poi/supermarkets4rich.csv', 'supermarkets4rich'),
    ('poi/workplaces.csv', 'workplaces')
]

# Список для сбора кусочков
all_pieces = []

for file, poi_type in files:
    print(f"Загружаем {file}...")
    
    # Читаем CSV
    df = pd.read_csv(file)
    
    # Проверяем наличие координат
    if not {'lat', 'lon'}.issubset(df.columns):
        print(f"Внимание: в {file} нет колонок lat/lon. Пропускаем.")
        continue
    
    # Создаем GeoDataFrame и добавляем тип точки
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lon, df.lat),
        crs="EPSG:4326"
    )
    
    # Добавляем колонку с типом POI для группировки
    gdf['poi_type'] = poi_type
    
    all_pieces.append(gdf)

# Объединяем все кусочки в один большой GeoDataFrame
gdf_all = gpd.GeoDataFrame(pd.concat(all_pieces, ignore_index=True), crs="EPSG:4326")
print("Единый датафрейм создан.")
print(f"Всего точек: {len(gdf_all)}")
fitness = pd.read_csv('Фитнес/fitness_w_centerd.csv')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами

WALK_SPEED_KMH = 4.5
METERS_PER_MIN = WALK_SPEED_KMH * 1000 / 60
WALK_TIME_MIN = 5
iso_type = 'ped'

# --- ЗАГРУЗКА ГРАФА (ОДИН РАЗ) ---
print("Загружаем пешеходный граф Москвы...")
# G_walk = ox.load_graphml(f'{GRAPH_PATH}/moscow_walk.graphml')
# G_walk = ox.project_graph(G_walk) # Проекция в метры (UTM)
# current_crs = G_walk.graph['crs']

start = time.time()  # или time.perf_counter()

# Проецируем офисы в ту же систему координат, что и граф
gdf_all = gdf_all.to_crs(current_crs)

# Добавляем атрибут времени на ребра графа (если его еще нет)
# Это нужно сделать один раз для всего графа, а не в цикле
for u, v, k, data in G_walk.edges(keys=True, data=True):
    # Проверяем наличие атрибута длины и приводим к float на всякий случай
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN

results = []
print(f"Начинаем обработку {len(fitness)} объектов...")


for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 1 == 250:
        print(f"Обрабатываем фитнес-клуб {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_walk, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_walk, center_node, radius=WALK_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась, записываем нули для всех типов
            empty_row = {'fitness_id': idx}
            for t in gdf_all['poi_type'].unique():
                empty_row[f'{t}_count_{WALK_TIME_MIN}{iso_type}'] = 0
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ (ГЛАВНАЯ ОПТИМИЗАЦИЯ) ---
        # Ищем все точки из ЕДИНОГО датафрейма, попавшие в полигон
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_all, how='inner', predicate='intersects')
        
        # Если ничего не найдено, записываем нули
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for t in gdf_all['poi_type'].unique():
                empty_row[f'{t}_count_{WALK_TIME_MIN}{iso_type}'] = 0
            results.append(empty_row)
            continue

# --- ГРУППИРОВКА И РАСЧЕТ МЕТРИК ---
        # Создаем словарь для хранения результатов по текущему фитнесу
        row_result = {'fitness_id': idx}
        
        # Группируем найденные точки по их типу
        grouped = hits.groupby('poi_type')

        for name, group in grouped:
            # 1. Считаем количество точек для каждого типа
            row_result[f'{name}_count_{WALK_TIME_MIN}{iso_type}'] = len(group)

            # 2. Считаем метрики для типов с рейтингами
            if name in ['beauty', 'cafe', 'coffee', 'dental', 'hospital', 
                        'parks', 'pharma', 'study', 'supermarkets', 
                        'supermarkets4rich', 'workplaces', 'molls']:
                # Проверяем, есть ли в этой группе колонка с отзывами и есть ли ненулевые значения
                if 'rating_cnt' in group.columns and group['rating_cnt'].notnull().any():
                    # .sum() и .mean() в pandas автоматически игнорируют NaN

                    row_result[f'{name}_reviews_sum_{WALK_TIME_MIN}{iso_type}'] = group['rating_cnt'].sum()
                    row_result[f'{name}_reviews_mean_{WALK_TIME_MIN}{iso_type}'] = group['rating_cnt'].mean()

            # 4. Специфичная логика для molls
            if name == 'molls':
                if 'branches_in_moll' in group.columns and group['branches_in_moll'].notnull().any():
                    row_result[f'molls_branches_sum_{WALK_TIME_MIN}{iso_type}'] = group['branches_in_moll'].sum()
                    row_result[f'molls_branches_mean_{WALK_TIME_MIN}{iso_type}'] = group['branches_in_moll'].mean()
                
                if 'building_floors' in group.columns and group['building_floors'].notnull().any():
                    row_result[f'molls_floors_sum_{WALK_TIME_MIN}{iso_type}'] = group['building_floors'].sum()
                    row_result[f'molls_floors_mean_{WALK_TIME_MIN}{iso_type}'] = group['building_floors'].mean()
                
                if 'parking' in group.columns and group['parking'].notnull().any():
                    row_result[f'molls_parking_sum_{WALK_TIME_MIN}{iso_type}'] = group['parking'].sum()
                    row_result[f'molls_parking_mean_{WALK_TIME_MIN}{iso_type}'] = group['parking'].mean()
        
        # Добавляем собранный результат в общий список
        results.append(row_result)
        

    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки тоже записываем нули для всех метрик, чтобы не ломать структуру финального датафрейма
        empty_row = {'fitness_id': idx}
        for t in gdf_all['poi_type'].unique():
            empty_row[f'{t}_count_{WALK_TIME_MIN}{iso_type}'] = 0.0
            empty_row[f'{t}_reviews_sum_{WALK_TIME_MIN}{iso_type}'] = 0.0
            empty_row[f'{t}_reviews_mean_{WALK_TIME_MIN}{iso_type}'] = 0.0
        results.append(empty_row)

Загружаем poi/beauty.csv...
Загружаем poi/bus.csv...
Загружаем poi/cafe.csv...
Загружаем poi/coffee.csv...
Загружаем poi/metro.csv...
Загружаем poi/molls.csv...
Загружаем poi/supermarkets4rich.csv...
Загружаем poi/workplaces.csv...
Единый датафрейм создан.
Всего точек: 43249
Загружаем пешеходный граф Москвы...
Начинаем обработку 5029 объектов...


In [60]:
res_df = pd.DataFrame(results)
print("Готово!")
end = time.time()
print(f"Время: {end - start:.4f} сек")

Готово!
Время: 17237.8535 сек


NameError: name 'fitness_poi_iso5_15ped' is not defined

In [62]:
17237/5029

3.4275203817856434

In [61]:
fitness_poi_iso5_15ped = pd.concat([new_df,res_df], axis=1)
fitness_poi_iso5ped = pd.concat([fitness,res_df], axis=1)

fitness_poi_iso5_15ped.to_csv('fitness_poi_iso5_15ped.csv', index = False)
fitness_poi_iso5_15ped.to_excel('fitness_poi_iso5_15ped.xlsx', index = False)

fitness_poi_iso5ped.to_csv('fitness_poi_iso5ped.csv', index = False)
fitness_poi_iso5ped.to_excel('fitness_poi_iso5ped.xlsx', index = False)

In [63]:
res_df.describe()

,fitness_id,beauty_count_5ped,beauty_reviews_sum_5ped,beauty_reviews_mean_5ped,bus_count_5ped,cafe_count_5ped,cafe_reviews_sum_5ped,cafe_reviews_mean_5ped,coffee_count_5ped,coffee_reviews_sum_5ped,...,metro_count_5ped,molls_count_5ped,molls_reviews_sum_5ped,molls_reviews_mean_5ped,molls_branches_sum_5ped,molls_branches_mean_5ped,molls_floors_sum_5ped,molls_floors_mean_5ped,molls_parking_sum_5ped,molls_parking_mean_5ped
count,5029.000000,4763.000000,4166.000000,4166.000000,4165.000000,3962.000000,3352.000000,3352.000000,3210.000000,2318.000000,...,982.000000,818.000000,305.000000,305.000000,303.000000,303.000000,303.000000,303.000000,305.000000,305.000000
mean,2514.000000,9.992022,288.788286,34.042229,2.136375,6.060828,619.315632,57.172741,2.528349,76.553926,...,0.623218,0.388753,239.314754,235.890164,177.257426,171.843234,6.399340,6.249175,624.360656,608.854098
std,1451.891582,11.816475,386.860711,45.693586,1.818529,9.847568,1509.762560,76.576318,2.817494,131.990568,...,0.771954,0.519364,349.832478,347.999965,142.278850,140.644817,5.554272,5.578886,987.336316,985.801035
min,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,1.000000,...,0.000000,0.000000,2.000000,2.000000,1.000000,1.000000,2.000000,2.000000,0.000000,0.000000
25%,1257.000000,3.000000,42.000000,11.000000,1.000000,1.000000,30.000000,14.741071,1.000000,6.000000,...,0.000000,0.000000,36.000000,34.000000,71.000000,71.000000,4.000000,4.000000,79.000000,79.000000
50%,2514.000000,7.000000,148.000000,23.333333,2.000000,3.000000,93.000000,32.000000,2.000000,26.000000,...,0.000000,0.000000,76.000000,76.000000,142.000000,136.000000,5.000000,4.000000,353.000000,322.000000
75%,3771.000000,13.000000,400.000000,41.860187,3.000000,6.000000,390.500000,67.750000,3.000000,91.000000,...,1.000000,1.000000,255.000000,255.000000,246.000000,208.000000,6.000000,6.000000,802.000000,802.000000
max,5028.000000,132.000000,3932.000000,637.000000,17.000000,80.000000,13622.000000,1157.000000,27.000000,1290.000000,...,5.000000,2.000000,1474.000000,1474.000000,613.000000,613.000000,40.000000,40.000000,6104.000000,6104.000000


In [64]:
fitness_poi_iso5_15ped.shape

(5029, 151)

In [65]:
fitness_poi_iso5_15ped[['beauty_count_5ped','beauty_count_15ped']]

,beauty_count_5ped,beauty_count_15ped
0,16.0,122.0
1,32.0,165.0
2,0.0,4.0
3,40.0,78.0
4,NaN,31.0
...,...,...
5024,12.0,48.0
5025,23.0,71.0
5026,16.0,37.0
5027,11.0,53.0


## Изохрона 15 мин на машине

In [2]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
from shapely.geometry import Point
import time
import numpy as np
import warnings
# Эта строка отключает отображение всех предупреждений DeprecationWarning
warnings.filterwarnings('ignore', category=DeprecationWarning)
# --- 1. ПОДГОТОВКА ЕДИНОГО ДАТАФРЕЙМА ---

# Список файлов и их типов
files = [
    ('poi/beauty.csv', 'beauty'),
    ('poi/bus.csv', 'bus'),
    ('poi/cafe.csv', 'cafe'),
    ('poi/cafe_checks.csv', 'cafe_checks'),
    ('poi/coffee.csv', 'coffee'),
    ('poi/dental.csv', 'dental'),
    ('poi/hospital.csv', 'hospital'),
    ('poi/metro.csv', 'metro'),
    ('poi/molls.csv', 'molls'),
    ('poi/parks.csv', 'parks'),
    ('poi/pharma.csv', 'pharma'),
    ('poi/study.csv', 'study'),
    ('poi/supermarkets.csv', 'supermarkets'),
    ('poi/supermarkets4rich.csv', 'supermarkets4rich'),
    ('poi/workplaces.csv', 'workplaces')
]


# Список для сбора кусочков
all_pieces = []

for file, poi_type in files:
    print(f"Загружаем {file}...")
    
    # Читаем CSV
    df = pd.read_csv(file)
    
    # Проверяем наличие координат
    if not {'lat', 'lon'}.issubset(df.columns):
        print(f"Внимание: в {file} нет колонок lat/lon. Пропускаем.")
        continue
    
    # Создаем GeoDataFrame и добавляем тип точки
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lon, df.lat),
        crs="EPSG:4326"
    )
    
    # Добавляем колонку с типом POI для группировки
    gdf['poi_type'] = poi_type
    
    all_pieces.append(gdf)

# Объединяем все кусочки в один большой GeoDataFrame
gdf_all = gpd.GeoDataFrame(pd.concat(all_pieces, ignore_index=True), crs="EPSG:4326")
print("Единый датафрейм создан.")
print(f"Всего точек: {len(gdf_all)}")
fitness = pd.read_csv('Фитнес/fitness_w_centerd.csv')

# --- НАСТРОЙКИ ---
GRAPH_PATH = r'D:/Users/Mariia/Desktop/Курсач/4 курс/2gis' # Путь к папке с графами

DRIVE_SPEED_KMH = 30 # Средняя скорость авто в городе
METERS_PER_MIN_DRIVE = DRIVE_SPEED_KMH * 1000 / 60
DRIVE_TIME_MIN = 15
iso_type = 'drive'
MAX_DRIVE_METERS = DRIVE_TIME_MIN * METERS_PER_MIN_DRIVE

# --- ЗАГРУЗКА ГРАФА (ОДИН РАЗ) ---
print("Загружаем дорожный граф Москвы...")
G_drive = ox.load_graphml(f'{GRAPH_PATH}/moscow_drive.graphml')
G_drive = ox.project_graph(G_drive) # Проекция в метры (UTM)
current_crs = G_drive.graph['crs']
start = time.time()  # или time.perf_counter()

# Проецируем офисы в ту же систему координат, что и граф
gdf_all = gdf_all.to_crs(current_crs)

# Добавляем атрибут времени на ребра графа (если его еще нет)
# Это нужно сделать один раз для всего графа, а не в цикле
for u, v, k, data in G_drive.edges(keys=True, data=True):
    # Проверяем наличие атрибута длины и приводим к float на всякий случай
    length = data.get('length')
    if length is not None:
        data['time'] = float(length) / METERS_PER_MIN_DRIVE

results = []
print(f"Начинаем обработку {len(fitness)} объектов...")


for idx, row in fitness.iterrows():
    lat, lon = row['lat'], row['lon']
    
    if idx % 250 == 0:
        print(f"Обрабатываем фитнес-клуб {idx+1}/{len(fitness)}...")
    
    try:
        # --- ПОСТРОЕНИЕ ИЗОХРОНЫ ---
        point_wgs84 = Point(lon, lat)
        point_graph_crs = gpd.GeoSeries([point_wgs84], crs="EPSG:4326").to_crs(current_crs).iloc[0]
        
        center_node = ox.distance.nearest_nodes(G_drive, X=point_graph_crs.x, Y=point_graph_crs.y)
        subgraph = nx.ego_graph(G_drive, center_node, radius=DRIVE_TIME_MIN, distance='time')
        
        if len(subgraph.nodes()) < 3:
            # Если изохрона не построилась, записываем нули для всех типов
            empty_row = {'fitness_id': idx}
            for t in gdf_all['poi_type'].unique():
                empty_row[f'{t}_count'] = 0
            results.append(empty_row)
            continue

        poly = gpd.GeoSeries([Point(data['x'], data['y']) for _, data in subgraph.nodes(data=True)]).unary_union.convex_hull

        # --- ПРОСТРАНСТВЕННОЕ СОЕДИНЕНИЕ (ГЛАВНАЯ ОПТИМИЗАЦИЯ) ---
        # Ищем все точки из ЕДИНОГО датафрейма, попавшие в полигон
        hits = gpd.sjoin(gpd.GeoDataFrame(geometry=[poly], crs=current_crs), gdf_all, how='inner', predicate='intersects')
        
        # Если ничего не найдено, записываем нули
        if hits.empty:
            empty_row = {'fitness_id': idx}
            for t in gdf_all['poi_type'].unique():
                empty_row[f'{t}_count'] = 0
            results.append(empty_row)
            continue

# --- ГРУППИРОВКА И РАСЧЕТ МЕТРИК ---
        # Создаем словарь для хранения результатов по текущему фитнесу
        row_result = {'fitness_id': idx}
        
        # Группируем найденные точки по их типу
        grouped = hits.groupby('poi_type')

        for name, group in grouped:
            # 1. Считаем количество точек для каждого типа
            row_result[f'{name}_count_{DRIVE_TIME_MIN}{iso_type}'] = len(group)

            # 2. Считаем метрики для типов с рейтингами
            if name in ['beauty', 'cafe', 'coffee', 'dental', 'hospital', 
                        'parks', 'pharma', 'study', 'supermarkets', 
                        'supermarkets4rich', 'workplaces', 'molls']:
                # Проверяем, есть ли в этой группе колонка с отзывами и есть ли ненулевые значения
                if 'rating_cnt' in group.columns and group['rating_cnt'].notnull().any():
                    # .sum() и .mean() в pandas автоматически игнорируют NaN
                    row_result[f'{name}_reviews_sum_{DRIVE_TIME_MIN}{iso_type}'] = group['rating_cnt'].sum()
                    row_result[f'{name}_reviews_mean_{DRIVE_TIME_MIN}{iso_type}'] = group['rating_cnt'].mean()

            # 3. Специфичная логика для cafe_checks (уникальные точки)
            if name == 'cafe_checks':
                # Группируем по координатам, чтобы получить уникальные кафе
                # Используем round(5) для сглаживания погрешностей координат
                unique_cafes = group.copy()
                unique_cafes['lat_round'] = unique_cafes['lat'].round(5)
                unique_cafes['lon_round'] = unique_cafes['lon'].round(5)
                unique_cafes = unique_cafes.groupby(['lat_round', 'lon_round'], as_index=False).first()

                # Считаем количество УНИКАЛЬНЫХ кафе
                row_result[f'cafe_checks_count_{DRIVE_TIME_MIN}{iso_type}'] = len(unique_cafes)

                # Считаем метрики только по уникальным записям
                if 'avg_price' in unique_cafes.columns and unique_cafes['avg_price'].notnull().any():
                    valid_prices = unique_cafes['avg_price'].dropna()
                    if len(valid_prices) > 0:
                        row_result[f'cafe_checks_avg_price_mean_{DRIVE_TIME_MIN}{iso_type}'] = valid_prices.mean()
                        row_result[f'cafe_checks_avg_price_median_{DRIVE_TIME_MIN}{iso_type}'] = valid_prices.median()

                if 'capacity' in unique_cafes.columns and unique_cafes['capacity'].notnull().any():
                    valid_capacity = unique_cafes['capacity'].dropna()
                    if len(valid_capacity) > 0:
                        row_result[f'cafe_checks_capacity_mean_{DRIVE_TIME_MIN}{iso_type}'] = valid_capacity.mean()
                        row_result[f'cafe_checks_capacity_median_{DRIVE_TIME_MIN}{iso_type}'] = valid_capacity.median()

            # 4. Специфичная логика для molls
            if name == 'molls':
                if 'branches_in_moll' in group.columns and group['branches_in_moll'].notnull().any():
                    row_result[f'molls_branches_sum_{DRIVE_TIME_MIN}{iso_type}'] = group['branches_in_moll'].sum()
                    row_result[f'molls_branches_mean_{DRIVE_TIME_MIN}{iso_type}'] = group['branches_in_moll'].mean()
                
                if 'building_floors' in group.columns and group['building_floors'].notnull().any():
                    row_result[f'molls_floors_sum_{DRIVE_TIME_MIN}{iso_type}'] = group['building_floors'].sum()
                    row_result[f'molls_floors_mean_{DRIVE_TIME_MIN}{iso_type}'] = group['building_floors'].mean()
                
                if 'parking' in group.columns and group['parking'].notnull().any():
                    row_result[f'molls_parking_sum_{DRIVE_TIME_MIN}{iso_type}'] = group['parking'].sum()
                    row_result[f'molls_parking_mean_{DRIVE_TIME_MIN}{iso_type}'] = group['parking'].mean()
     
        # Добавляем собранный результат в общий список
        results.append(row_result)
        

    except Exception as e:
        print(f"Ошибка на индексе {idx}: {e}")
        # В случае ошибки тоже записываем нули для всех метрик, чтобы не ломать структуру финального датафрейма
        empty_row = {'fitness_id': idx}
        for t in gdf_all['poi_type'].unique():
            empty_row[f'{t}_count_{DRIVE_TIME_MIN}{iso_type}'] = 0.0
            empty_row[f'{t}_reviews_sum_{DRIVE_TIME_MIN}{iso_type}'] = 0.0
            empty_row[f'{t}_reviews_mean_{DRIVE_TIME_MIN}{iso_type}'] = 0.0
        results.append(empty_row)

Загружаем poi/beauty.csv...
Загружаем poi/bus.csv...
Загружаем poi/cafe.csv...
Загружаем poi/cafe_checks.csv...
Загружаем poi/coffee.csv...
Загружаем poi/dental.csv...
Загружаем poi/hospital.csv...
Загружаем poi/metro.csv...
Загружаем poi/molls.csv...
Загружаем poi/parks.csv...
Загружаем poi/pharma.csv...
Загружаем poi/study.csv...
Загружаем poi/supermarkets.csv...
Загружаем poi/supermarkets4rich.csv...
Загружаем poi/workplaces.csv...
Единый датафрейм создан.
Всего точек: 72436
Загружаем дорожный граф Москвы...
Начинаем обработку 5029 объектов...
Обрабатываем фитнес-клуб 1/5029...
Обрабатываем фитнес-клуб 251/5029...
Обрабатываем фитнес-клуб 501/5029...
Обрабатываем фитнес-клуб 751/5029...
Обрабатываем фитнес-клуб 1001/5029...
Обрабатываем фитнес-клуб 1251/5029...
Обрабатываем фитнес-клуб 1501/5029...
Обрабатываем фитнес-клуб 1751/5029...
Обрабатываем фитнес-клуб 2001/5029...
Обрабатываем фитнес-клуб 2251/5029...
Обрабатываем фитнес-клуб 2501/5029...
Обрабатываем фитнес-клуб 2751/5029.

In [3]:
res_df = pd.DataFrame(results)
print("Готово!")
end = time.time()
print(f"Время: {end - start:.4f} сек")

Готово!
Время: 1941.9371 сек


In [7]:
1941/60/60

0.5391666666666667

In [5]:
fitness_poi_iso5_15ped = pd.read_csv('fitness_poi_iso5_15ped.csv')

fitness_poi_iso5_15ped_15drive = pd.concat([fitness_poi_iso5_15ped,res_df], axis=1)
fitness_poi_iso15drive = pd.concat([fitness,res_df], axis=1)

fitness_poi_iso5_15ped_15drive.to_csv('fitness_poi_iso5_15ped_15drive.csv', index = False)
fitness_poi_iso5_15ped_15drive.to_excel('fitness_poi_iso5_15ped_15drive.xlsx', index = False)

fitness_poi_iso15drive.to_csv('fitness_poi_iso15drive.csv', index = False)
fitness_poi_iso15drive.to_excel('fitness_poi_iso15drive.xlsx', index = False)

In [6]:
fitness_poi_iso5_15ped_15drive[['beauty_count_5ped','beauty_count_15ped', 'beauty_count_15drive']]

,beauty_count_5ped,beauty_count_15ped,beauty_count_15drive
0,16.0,122.0,4149.0
1,32.0,165.0,3619.0
2,0.0,4.0,2638.0
3,40.0,78.0,495.0
4,NaN,31.0,2982.0
...,...,...,...
5024,12.0,48.0,851.0
5025,23.0,71.0,646.0
5026,16.0,37.0,759.0
5027,11.0,53.0,749.0
